## A/B testing (initial/exploratory hypothesis)

In [5]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [6]:

df_grocery = pd.read_csv("data/df_grocery_cleaned.csv")
df_grocery = df_grocery.copy()


In [7]:
df_grocery.head()

,order_id,agent_age,agent_rating,store_latitude,store_longitude,drop_latitude,drop_longitude,order_date,order_time,pickup_time,...,category,pickup_hour,weekday,week,distance,age_group,order_datetime,pickup_datetime,time_of_day,order_to_pickup_min
0,uhfs888375680,35,4.0,12.311072,76.654878,12.351072,76.694878,2022-03-01,14:55:00,15:10:00,...,Grocery,15,Tuesday,9,6.218001,26-35,2022-03-01 14:55:00,2022-03-01 15:10:00,Afternoon,15.0
1,zotl583816092,32,3.5,0.000000,0.000000,0.110000,0.110000,2022-03-08,21:35:00,21:45:00,...,Grocery,21,Tuesday,10,17.297866,26-35,2022-03-08 21:35:00,2022-03-08 21:45:00,Evening,10.0
2,zkat983341391,35,4.3,13.064181,80.236442,13.134181,80.306442,2022-03-14,17:25:00,17:30:00,...,Grocery,17,Monday,11,10.865465,26-35,2022-03-14 17:25:00,2022-03-14 17:30:00,Afternoon,5.0
3,nvck471535045,37,4.9,22.539129,88.365507,22.559129,88.385507,2022-02-13,10:55:00,11:00:00,...,Grocery,11,Sunday,6,3.027237,36-45,2022-02-13 10:55:00,2022-02-13 11:00:00,Morning,5.0
4,emlt327861941,23,4.8,0.000000,0.000000,0.020000,0.020000,2022-03-07,10:40:00,10:50:00,...,Grocery,10,Monday,10,3.145067,18-25,2022-03-07 10:40:00,2022-03-07 10:50:00,Morning,10.0


## 🎯 A/B Test: Traffic-Aware Dispatch

- 🅰️ Control → proximity-based allocation (baseline)
- 🅱️ Treatment → traffic-performance-based allocation


In [8]:
# df_grocery.head()

# for col in df_grocery.columns:
#     print(col)



In [9]:
np.random.seed(42)  # para reprodutibilidade

df_grocery["group"] = np.random.choice(
    ["control", "treatment"],
    size=len(df_grocery),
    p=[0.5, 0.5]
)


In [10]:
# df_grocery.head()

### sanity check / balance check

In [11]:
df_grocery["group"].value_counts()

group
treatment    1353
control      1335
Name: count, dtype: int64

In [12]:
counts = df_grocery["group"].value_counts(normalize=True) * 100

control_pct = counts.get("control", 0)
treatment_pct = counts.get("treatment", 0)

print(f"Control: {control_pct:.1f}%")
print(f"Treatment: {treatment_pct:.1f}%")

if abs(control_pct - 50) <= 5 and abs(treatment_pct - 50) <= 5:
    print("Great, it's balanced ✅")
else:
    print("Warning: groups may be imbalanced ⚠️")

Control: 49.7%
Treatment: 50.3%
Great, it's balanced ✅


The randomization produced well-balanced groups in terms of sample size, which is the first sanity check for a valid A/B test.

### Are the groups similar in important characteristics?

uplift = [(treatment - control)/control] x 100

In [13]:
df_grocery.groupby("group")["distance"].mean()

group
control      23.24546
treatment    25.84260
Name: distance, dtype: float64

In [14]:
df_grocery.groupby("group")["delivery_time"].mean()

group
control      26.625468
treatment    26.457502
Name: delivery_time, dtype: float64

In [15]:
control_mean = df_grocery[df_grocery["group"] == "control"]["delivery_time"].mean()
treatment_mean = df_grocery[df_grocery["group"] == "treatment"]["delivery_time"].mean()

uplift = ((treatment_mean - control_mean) / control_mean) * 100

print(uplift)

-0.6308483141353891


🔻 negative = improved (because it reduced the time)

📉 -0.63% = slight improvement

The new allocation model reduced the average delivery time by about 0.6% compared to the current proximity-based model.

### Is this real, or could it have happened by chance? **statistical significance (t-test)**

While the treatment group handled slightly longer distances, delivery times were marginally lower, suggesting potential efficiency gains. However, the effect size is small and requires statistical validation.

The t-statistic measures the magnitude of the difference between groups relative to variability, while the p-value indicates the probability that this difference is due to random chance. **In my experiment, the high p-value suggests the observed uplift is not statistically significant.**

t-stat → “how different are the weights?”

p-value → “can I trust this difference, or was it just luck?”

In [16]:
# to the metric delivery_time
control = df_grocery[df_grocery["group"] == "control"]["delivery_time"].dropna()
treatment = df_grocery[df_grocery["group"] == "treatment"]["delivery_time"].dropna()


In [17]:
from scipy.stats import ttest_ind

t_stat, p_value = ttest_ind(control, treatment)

print("t-statistic:", t_stat)
print("p-value:", p_value)

t-statistic: 0.4582578002860262
p-value: 0.6468043100682419


**Results of the A/B testing:**

The A/B test compared a proximity-based assignment strategy (control) with a traffic-aware assignment strategy (treatment). 

The results showed a very small reduction in delivery time for the treatment group (approximately -0.63% uplift). However, the difference was not statistically significant (p-value = 0.64), indicating that we cannot conclude a measurable improvement from the new dispatch logic. This suggests that, in this dataset, the proposed optimization did not have a strong enough impact to outperform the baseline, and observed differences are likely due to random variation.

### Power Analysis

**What is the probability that my test will detect a real effect, if one exists?**

Formula:  **Power= 1 - "beta"**

Classic guideline (Cohen's rule): 
**d ≈ 0.2 (small), 0.5 (medium), 0.8 (large)**

Results: **There is no significant difference between the control and treatment groups in terms of standardized effect size**

In [18]:
from statsmodels.stats.power import TTestIndPower

analysis = TTestIndPower()

In [19]:
# Cohen’s d

effect_size = (treatment.mean() - control.mean()) / control.std()
print(effect_size)

-0.017682043069599952


The effect size (Cohen’s d ≈ -0.018) indicates a negligible standardized difference between control and treatment, which aligns with the non-significant p-value, suggesting no meaningful impact of the treatment.

Based on the experiment results, **the traffic-aware dispatch strategy did not show a statistically significant improvement in delivery time, with a negligible effect size**. Therefore, I would not ship this change. Instead, **I would iterate on the hypothesis, potentially by segmenting the analysis by peak traffic hours or refining the definition of rider performance under congestion**.

We are not deciding whether the test is valid; we are deciding whether to implement the treatment in production.